In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

# -------- CONFIG --------
SOURCE = "C:/Users/soura/Downloads/sb/Channel ID_18_3.mp4"  # or RTSP
MIN_AREA = 1500
FRAME_SKIP = 2
YOLO_CONF = 0.3

# -------- INIT --------
cap = cv2.VideoCapture(SOURCE)
model = YOLO("yolov8n.pt")

ret, prev_frame = cap.read()
if not ret:
    raise Exception("Cannot read source")

prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
prev_gray = cv2.GaussianBlur(prev_gray, (21, 21), 0)

frame_id = 0

# -------- LOOP --------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1
    if frame_id % FRAME_SKIP != 0:
        continue

    # ---- MOTION DETECTION ----
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (21, 21), 0)

    delta = cv2.absdiff(prev_gray, gray)
    thresh = cv2.threshold(delta, 25, 255, cv2.THRESH_BINARY)[1]
    thresh = cv2.dilate(thresh, None, iterations=2)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    motion_boxes = []

    for cnt in contours:
        if cv2.contourArea(cnt) < MIN_AREA:
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        motion_boxes.append((x, y, w, h))

    # ---- AGGREGATE INTO SINGLE ROI ----
    aggregated_box = None
    if motion_boxes:
        x_min = min([x for x, y, w, h in motion_boxes])
        y_min = min([y for x, y, w, h in motion_boxes])
        x_max = max([x+w for x, y, w, h in motion_boxes])
        y_max = max([y+h for x, y, w, h in motion_boxes])

        aggregated_box = (x_min, y_min, x_max - x_min, y_max - y_min)

        # draw motion ROI (blue)
        cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (255, 0, 0), 2)

    # ---- YOLO ON ROI ----
    if aggregated_box:
        x, y, w, h = aggregated_box

        # clip bounds
        h_frame, w_frame = frame.shape[:2]
        x = max(0, x)
        y = max(0, y)
        w = min(w, w_frame - x)
        h = min(h, h_frame - y)

        roi = frame[y:y+h, x:x+w]

        if roi.size > 0:
            results = model(roi, classes=[0], conf=YOLO_CONF)

            for r in results:
                boxes = r.boxes.xyxy.cpu().numpy()
                scores = r.boxes.conf.cpu().numpy()

                for i, box in enumerate(boxes):
                    if scores[i] < YOLO_CONF:
                        continue

                    x1, y1, x2, y2 = box

                    # map back to full frame
                    x1_full = int(x + x1)
                    y1_full = int(y + y1)
                    x2_full = int(x + x2)
                    y2_full = int(y + y2)

                    # draw detection (green)
                    cv2.rectangle(frame, (x1_full, y1_full),
                                  (x2_full, y2_full), (0, 255, 0), 2)

    prev_gray = gray

    cv2.imshow("frame", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()